In [1]:
import matplotlib.pyplot as plt
from vrae.vrae import VRAE
import time
import shutil
import numpy as np
import torch
import torch.nn as nn   
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
## trainig curves
plot_until_epoch = 200
n_replicate = 5
beta_range = ['0','0.1','0.5','1.0','2.0','5.0']
path=fr'C:\Users\achfr\timeseries-clustering-vae\beta_sweep_1'
loss_dict = {}
for beta in beta_range:
    loss_dict[beta] = ([],[])
    for replicate in range(n_replicate):
        train_losses = np.load(fr'{path}\train_losses_{replicate}_{beta}.npy')
        test_losses = np.load(fr'{path}\test_losses_{replicate}_{beta}.npy')
        loss_dict[beta][0].append(train_losses)
        loss_dict[beta][1].append(test_losses)
# plot mean train loss and mean test loss with std curves+fill as a function of latent dimension using loss_dict 
mean_train_losses = np.zeros((len(beta_range),plot_until_epoch))
mean_test_losses = np.zeros((len(beta_range),plot_until_epoch))
std_train_losses = np.zeros((len(beta_range),plot_until_epoch))
std_test_losses = np.zeros((len(beta_range),plot_until_epoch))

for i,beta in enumerate(beta_range):
    mean_train_losses[i] = np.mean([arr[:plot_until_epoch] for arr in loss_dict[beta][0]], axis=0)
    mean_test_losses[i] = np.mean([arr[:plot_until_epoch] for arr in loss_dict[beta][1]], axis=0)
    std_train_losses[i] = np.std([arr[:plot_until_epoch] for arr in loss_dict[beta][0]], axis=0)
    std_test_losses[i] = np.std([arr[:plot_until_epoch] for arr in loss_dict[beta][1]], axis=0)

fig,ax = plt.subplots(1,2,figsize=(27,8))
cmap = plt.get_cmap('jet')
colors = [cmap(x) for x in np.linspace(0, 0.9, len(beta_range))]
for i,beta in enumerate(beta_range):
    ax[0].plot(mean_train_losses[i,:plot_until_epoch], label=f'train beta {beta}', color=colors[i])
    ax[0].fill_between(range(plot_until_epoch), 
                    mean_train_losses[i,:plot_until_epoch]-std_train_losses[i,:plot_until_epoch], 
                    mean_train_losses[i,:plot_until_epoch]+std_train_losses[i,:plot_until_epoch], 
                    color=colors[i], alpha=0.3)
    ax[1].plot(mean_test_losses[i,:plot_until_epoch], label=f'test beta {beta}', color=colors[i])
    ax[1].fill_between(range(plot_until_epoch), 
                    mean_test_losses[i,:plot_until_epoch]-std_test_losses[i,:plot_until_epoch], 
                    mean_test_losses[i,:plot_until_epoch]+std_test_losses[i,:plot_until_epoch], 
                    color=colors[i], alpha=0.3)
    ax[0].set_title('Train Losses with Std Dev')
    ax[1].set_title('Test Losses with Std Dev')
    ax[0].set_xlabel('Epochs')
    ax[1].set_xlabel('Epochs')
    ax[0].set_ylabel('Loss')
plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1))

# plot best test loss with std curves+fill as a function of latent dimension
std_test_losses = []
best_test_loss = []
for beta in beta_range:
    test_losses = [arr[:plot_until_epoch] for arr in loss_dict[beta][1]]
    best_losses = np.min(np.array(test_losses)[:,-1])
    std_loss = np.std(np.array(test_losses)[:,-1])
    best_test_loss.append(best_losses)
    std_test_losses.append(std_loss)


plt.figure(figsize=(10,6))
colors = {'0':'blue', '0.1':'orange', '0.5':'green', '1.0':'red', '2.0':'purple', '5.0':'brown'}

for i,beta in enumerate(beta_range):

    # Normalize to 0-1 range
    best_test_loss_arr = np.array(best_test_loss)
    std_test_losses_arr = np.array(std_test_losses)
    
    min_loss = np.min(best_test_loss_arr)
    max_loss = np.max(best_test_loss_arr)
    
    normalized_loss = (best_test_loss_arr - min_loss) / (max_loss - min_loss)
    normalized_std = std_test_losses_arr / (max_loss - min_loss)
    
    plt.plot(beta_range, normalized_loss, marker='x', label=f'Beta={beta}', color=colors[beta])
    plt.fill_between(beta_range, 
                     normalized_loss - normalized_std, 
                     normalized_loss + normalized_std, 
                     alpha=0.2, color=colors[beta])
plt.legend()
plt.xlabel('Beta')
plt.ylabel('Normalized Best Test Loss')
# plt.title('Normalized Test Loss vs Latent Dimension by Condition')
plt.grid(True, alpha=0.3)
plt.show()

In [26]:
X_train = np.load('numpy_save_files/X_train_growth_antibiotic.npy', allow_pickle=True)
y_train = np.load('numpy_save_files/y_train_growth_antibiotic.npy', allow_pickle=True)
X_test = np.load('numpy_save_files/X_test_growth_antibiotic.npy', allow_pickle=True)
y_test = np.load('numpy_save_files/y_test_growth_antibiotic.npy', allow_pickle=True)

print('X_train shape\t',X_train.shape,'\t X_test shape\t',X_test.shape)

train_dataset = TensorDataset(torch.from_numpy(X_train).to('cuda'))
test_dataset = TensorDataset(torch.from_numpy(X_test))

## load model 
latent_length = 12
sequence_length = 72
number_of_features = 1
best_replicate = 0
dload = './model_dir_mm' #download directory
hidden_size = 90
hidden_layer_depth = 2
batch_size = 200
learning_rate = 0.001 # 0.0005
n_epochs = 200
dropout_rate = 0.2
optimizer = 'Adam' # options: ADAM, SGD
cuda = True # options: True, False
print_every=1000
clip = True # options: True, False
max_grad_norm=5
loss = 'MSELoss' # options: SmoothL1Loss, MSELoss
block = 'LSTM' # options: LSTM, GRU
beta = '0' # options: '0','0.1','0.5','1.0','2.0','5.0'

vrae_beta= VRAE(sequence_length=sequence_length,
        number_of_features = number_of_features,
        hidden_size = hidden_size, 
        hidden_layer_depth = hidden_layer_depth,
        latent_length = latent_length,
        batch_size = batch_size,
        learning_rate = learning_rate,
        n_epochs = n_epochs,
        dropout_rate = dropout_rate,
        optimizer = optimizer, 
        cuda = cuda,
        print_every=print_every, 
        clip=clip, 
        max_grad_norm=max_grad_norm,
        loss = loss,
        block = block,
        dload = dload,
        beta=beta)

vrae_beta.load(fr'C:\Users\achfr\timeseries-clustering-vae\beta_sweep_1\model_best_0_0.pth')
# vrae_beta.eval()

X_train shape	 (57607, 72, 1) 	 X_test shape	 (699, 72, 1)


In [29]:
## load model 
sequence_length = 72
number_of_features = 1
best_replicate = 0

dload = './model_dir_mm' #download directory
hidden_size = 90
hidden_layer_depth = 2
batch_size = 100
learning_rate = 0.0005 # 0.0005
n_epochs = 400
dropout_rate = 0.2
optimizer = 'Adam' # options: ADAM, SGD
cuda = True # options: True, False
print_every=1000
clip = True # options: True, False
max_grad_norm=5
loss = 'MSELoss' # options: SmoothL1Loss, MSELoss
block = 'LSTM' # options: LSTM, GRU

vrae_trained_size = VRAE(sequence_length=sequence_length,
        number_of_features = number_of_features,
        hidden_size = hidden_size, 
        hidden_layer_depth = hidden_layer_depth,
        latent_length = latent_length,
        batch_size = batch_size,
        learning_rate = learning_rate,
        n_epochs = n_epochs,
        dropout_rate = dropout_rate,
        optimizer = optimizer, 
        cuda = cuda,
        print_every=print_every, 
        clip=clip, 
        max_grad_norm=max_grad_norm,
        loss = loss,
        block = block,
        dload = dload)

vrae_trained_size.load(fr'C:\Users\achfr\timeseries-clustering-vae\multivar_72_size\model_best_{best_replicate}_{latent_length}.pth')


vrae_trained_size.eval()
# testseq2 = test_size[:batch_size][0].float().permute(1, 0, 2).cuda()
# print(testseq2.shape)

# outp = vrae_trained_size.forward(testseq2)

AttributeError: 'VRAE' object has no attribute 'block'

AttributeError: 'VRAE' object has no attribute 'block'

VRAE(n_epochs=400,batch_size=100,cuda=True)